Day 3, Topic 4: Universal Functions (Ufuncs) and Broadcasting in Action

## 📘 Day 3, Topic 4: Universal Functions (Ufuncs) and Broadcasting in Action

### What Are Ufuncs?
**Universal Functions (ufuncs)** are NumPy's compiled C-level functions that operate **element-wise** on arrays. Every arithmetic operator (`+`, `*`, etc.) is backed by a ufunc. They're fast because they:
- Run in compiled C/Fortran (no Python interpreter overhead per element)
- Support broadcasting natively
- Have extra features: `out=`, `.reduce()`, `.accumulate()`, `.outer()`, `.at()`

Examples: `np.add`, `np.multiply`, `np.sqrt`, `np.exp`, `np.sin`, `np.maximum`

---

### `out=` Parameter — Zero Extra Memory Allocation
Instead of creating a new array for the result, write directly into a **pre-allocated** array:
```python
a = np.arange(1_000_000)
b = np.arange(1_000_000)
result = np.empty_like(a)     # allocate once
np.add(a, b, out=result)      # no extra allocation during computation
```
> 💡 Use `out=` in performance-critical loops to avoid repeated memory allocation. Measured speedup is real — see Task 4 below.

---

### `.reduce()` — Collapse an Axis
Repeatedly applies the ufunc to collapse an array along an axis (like `np.sum` but generalized):
```python
arr = np.array([1, 2, 3, 4])
np.add.reduce(arr)        # → 10   (1+2+3+4)
np.multiply.reduce(arr)   # → 24   (1×2×3×4)
```

For 2D arrays:
```python
# axis=1 → reduce across columns (one result per row)
np.add.reduce(arr_2d, axis=1)   # row sums

# axis=0 → reduce down rows (one result per column)
np.add.reduce(arr_2d, axis=0)   # column sums
```

For 3D arrays `(2, 3, 4)`:
```
axis=0 → collapses blocks   → shape (3, 4)
axis=1 → collapses rows     → shape (2, 4)
axis=2 → collapses columns  → shape (2, 3)
```

---

### `.accumulate()` — Running Cumulative Result
Like `.reduce()` but **keeps all intermediate results** (equivalent to `np.cumsum`/`np.cumprod`):
```python
arr = np.array([1, 2, 3, 4])
np.add.accumulate(arr)        # → [1, 3, 6, 10]   running sum
np.multiply.accumulate(arr)   # → [1, 2, 6, 24]   running product
```

For 2D arrays:
```python
# axis=0 → running sum down each column
np.add.accumulate(arr_2d, axis=0)
# [[1, 2], [4, 6]]   row 1 accumulates row 0

# axis=1 → running sum across each row
np.add.accumulate(arr_2d, axis=1)
# [[1, 3], [3, 7]]
```

---

### `.outer()` — All Pairwise Combinations
Computes the operation for **every combination** of elements from two arrays:
```python
a = np.array([1, 2, 3])
b = np.array([4, 5])
np.multiply.outer(a, b)
# → [[4,  5],     shape (3, 2)
#    [8, 10],
#    [12, 15]]
```
Equivalent to: `a[:, np.newaxis] * b` — but `.outer()` works for any ufunc.

---

### `.reduceat()` — Segmented Reduction
Reduces different **slices** of the array independently. Useful for group-wise aggregation:
```python
arr = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
indices = [0, 2, 5]   # defines slice starts: [0:2], [2:5], [5:end]
np.add.reduceat(arr, indices)
# → [3, 12, 40]    sum of each segment
```

---

### `np.frompyfunc()` — Wrap a Python Function as a Ufunc
Turn any Python function into a ufunc so it broadcasts over arrays:
```python
def celsius_to_f(c):
    return c * 9/5 + 32

c_to_f = np.frompyfunc(celsius_to_f, 1, 1)   # 1 input, 1 output
c_to_f(np.array([0, 10, 20, 30]))            # → [32, 50, 68, 86]
```
> ⚠️ `np.frompyfunc()` returns dtype `object`. For typed output, use `np.vectorize(...).astype(float)` or write the vectorized math directly.

---

### Casting — Controlling Output dtype
```python
a = np.array([1, 2, 3], dtype=np.int32)
b = np.array([2.2, 4.3, 5.6], dtype=np.float64)

np.add(a, b)               # → float64 (upcasted automatically)
np.add(a, b, dtype=np.float32)  # force output to float32
```

---

### `keepdims=True` — Preserve Shape for Broadcasting
When computing a statistic along an axis, use `keepdims=True` so the result stays broadcastable:
```python
arr = np.random.rand(3, 4)
mean = arr.mean(axis=1, keepdims=True)   # shape (3, 1) — NOT (3,)
arr - mean                               # (3,4) - (3,1) → (3,4) ✓

# Without keepdims:
mean_bad = arr.mean(axis=1)             # shape (3,) → can't subtract from (3,4) directly
```

In [2]:
import numpy as np

In [3]:
#out parameter
a = np.arange(1_000_000)
b = np.arange(1_000_000)
result = np.empty_like(a)
np.add(a, b, out=result)

array([      0,       2,       4, ..., 1999994, 1999996, 1999998],
      shape=(1000000,))

In [11]:
#.reduce: collapse an array along an axis
#1D array
arr = np.array([1, 2, 3, 4])
print(np.add.reduce(arr))
print(np.multiply.reduce(arr))

10
24


In [14]:
#2d array
arr_2d = np.array([[1, 2, 3],
                   [4, 5, 6]])
print(np.add.reduce(arr_2d, axis=1)) #sum across column
print(np.add.reduce(arr_2d, axis=0)) #sum down row

[ 6 15]
[5 7 9]


In [15]:
#3d array
arr_3d = np.arange(24).reshape(2, 3, 4)
print(arr_3d)

[[[ 0  1  2  3]
  [ 4  5  6  7]
  [ 8  9 10 11]]

 [[12 13 14 15]
  [16 17 18 19]
  [20 21 22 23]]]


In [19]:
print(np.add.reduce(arr_3d, axis=0))
print(np.add.reduce(arr_3d, axis=1))
print(np.add.reduce(arr_3d, axis=2))

[[12 14 16 18]
 [20 22 24 26]
 [28 30 32 34]]
[[12 15 18 21]
 [48 51 54 57]]
[[ 6 22 38]
 [54 70 86]]


In [21]:
#.accumulate() – Running Total (Cumulative)
arr = np.array([1, 2, 3, 4])
print(np.add.accumulate(arr))
print(np.multiply.accumulate(arr))

[ 1  3  6 10]
[ 1  2  6 24]


In [24]:
arr_2d = np.array([[1, 2],
                   [3, 4]])
print(np.add.accumulate(arr_2d, axis=0))
print(np.add.accumulate(arr_2d, axis=1))

[[1 2]
 [4 6]]
[[1 3]
 [3 7]]


In [25]:
arr_3d = np.ones((2, 3, 4))
print(arr_3d)

[[[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]

 [[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]]


In [27]:
print(np.add.accumulate(arr_3d, axis=0))
print(np.add.accumulate(arr_3d, axis=1))
print(np.add.accumulate(arr_3d, axis=2))

[[[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]

 [[2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]]]
[[[1. 1. 1. 1.]
  [2. 2. 2. 2.]
  [3. 3. 3. 3.]]

 [[1. 1. 1. 1.]
  [2. 2. 2. 2.]
  [3. 3. 3. 3.]]]
[[[1. 2. 3. 4.]
  [1. 2. 3. 4.]
  [1. 2. 3. 4.]]

 [[1. 2. 3. 4.]
  [1. 2. 3. 4.]
  [1. 2. 3. 4.]]]


In [3]:
#outer() - All pairs combinations
a = np.array([1, 2, 3])
b = np.array([4, 5])
print(np.multiply.outer(a, b))

[[ 4  5]
 [ 8 10]
 [12 15]]


In [4]:
#.reduceat() – Segmented Reductions
arr = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
indices = [0, 2, 5] # Start: [0:2], then: [2:5], and end at: [5:end]
print(np.add.reduceat(arr, indices))

[ 3 12 40]


In [5]:
def celsius_to_fahrenhiet(c):
    return c * 9/5 + 32

c_to_f_ufunc = np.frompyfunc(celsius_to_fahrenhiet, 1, 1)
temps = np.array([0, 10, 20, 30])
temps_f = c_to_f_ufunc(temps)
print(temps_f)

[32.0 50.0 68.0 86.0]


In [6]:
temps_f.dtype

dtype('O')

In [9]:
a = np.array([1, 2, 3], dtype=np.int32)
b = np.array([2.2, 4.3, 5.6], dtype=np.float64)
c = np.add(a, b, dtype=np.float32)

In [10]:
c.dtype

dtype('float32')

In [14]:
arr = np.random.rand(3, 4)
mean_rows = arr.mean(axis=1, keepdims=True)
result = arr - mean_rows

In [15]:
arr

array([[0.46455981, 0.64810838, 0.32476358, 0.30745819],
       [0.57782955, 0.90138283, 0.34587419, 0.82736282],
       [0.9357972 , 0.33420651, 0.5350153 , 0.1648387 ]])

In [16]:
mean_rows

array([[0.43622249],
       [0.66311235],
       [0.49246443]])

In [17]:
result

array([[ 0.02833732,  0.21188589, -0.11145891, -0.1287643 ],
       [-0.0852828 ,  0.23827048, -0.31723815,  0.16425047],
       [ 0.44333277, -0.15825792,  0.04255087, -0.32762573]])

In [18]:
#Practice Activity: Ufuncs and Broadcasting

In [21]:
# Task 1: Given a = np.arange(1, 5) and b = np.arange(1, 4), compute the outer product 
#using np.multiply.outer. Verify it matches a[:, np.newaxis] * b.
a = np.arange(1, 5) #1, 2, 3, 4
b = np.arange(1, 4) # 1, 2, 3
result_1 = np.multiply.outer(a, b)
result_2 = a[:, np.newaxis] * b
print(result_1)

[[ 1  2  3]
 [ 2  4  6]
 [ 3  6  9]
 [ 4  8 12]]


In [24]:
#Task 2: Given a 3D array data = np.random.rand(5, 10, 10) (5 images of 10x10), 
#use np.add.accumulate to compute the cumulative sum along the rows (axis=1) for 
#each image. Check shape.
data = np.random.rand(5, 10, 10)
result = np.add.accumulate(data, axis=1)
print(result.shape)

(5, 10, 10)


In [41]:
#Task 3: Compute the product of all elements in a 3D array (2,3,4) 
#using np.multiply.reduce with axis=None. Compare with arr.prod().
arr_3d = np.arange(1, 25).reshape(2, 3, 4)
print("Original shape:", arr_3d.shape)
# Correct version for all elements
result_all = np.multiply.reduce(arr_3d, axis=None)
print("Product of all elements:", result_all)
print("Using .prod():", arr_3d.prod())

Original shape: (2, 3, 4)
Product of all elements: -7835185981329244160
Using .prod(): -7835185981329244160


In [35]:
# Task 4: Create two large arrays x = np.ones(1_000_000) and y = np.ones(1_000_000). 
#Use np.add with out to store the sum in a pre‑allocated array z. 
#Time it vs. z = x + y using %timeit. Observe memory allocation.
x = np.ones(1_000_000)
y = np.ones(1_000_000)
z = np.empty_like(x)
%timeit np.add(x, y, out=z)
%timeit x + y

1.89 ms ± 40.3 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
3.5 ms ± 77.9 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [40]:
#Task 5: Write a custom ufunc using np.frompyfunc that computes f(x) = 1 / (1 + exp(-x)) 
#(sigmoid). Apply it to an array. Then compare its speed with 
#the vectorized version 1 / (1 + np.exp(-x)).

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

sigmoid_cal = np.frompyfunc(sigmoid, 1, 1)
arr = np.array([-10, 20, 40, 30])
%timeit result = sigmoid_cal(arr)
%timeit 1 / (1 + np.exp(-arr))

10.4 μs ± 487 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
4.95 μs ± 138 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
